# SMART Baseline Training (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart-baseline_colab.ipynb)

## Objective
- Train SMART baseline and persist checkpoints to Drive.
- Keep this notebook training-only.
- Run simulation/evaluation in dedicated eval/comparative notebooks.


In [ ]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-baseline config and define checkpoint path
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-baseline"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_baseline"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

BASELINE_CKPT_DIR = os.environ.get("SMART_BASELINE_CKPT_DIR", "").strip() or str(persist_run_dir / "checkpoints" / "smart_baseline")
print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("BASELINE_CKPT_DIR:", BASELINE_CKPT_DIR)
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Build SMART baseline training command plan (train-only)
from src.workflows import run_smart_baseline_flow

flow_bundle = run_smart_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path="",
    smart_save_ckpt_path=BASELINE_CKPT_DIR,
    smart_raw_data_root=str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART.get("install_pyg", True)),
    sync_smart_repo=True,
)

print("flow_summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("train_cmd:")
print(flow_bundle.command_plan["train_cmd"])
print("validate_cmd (not used in this notebook):")
print(flow_bundle.command_plan["validate_cmd"])


In [ ]:
# Step 4: Optional training execution (no evaluation in this notebook)
RUN_SETUP = False
RUN_PREPROCESS = False
RUN_TRAIN = False

cmds = []
if RUN_SETUP:
    cmds.append(flow_bundle.command_plan["setup_cmd"])
if RUN_PREPROCESS:
    cmds.extend([
        flow_bundle.command_plan["preprocess_train_cmd"],
        flow_bundle.command_plan["preprocess_val_cmd"],
    ])
if RUN_TRAIN:
    cmds.append(flow_bundle.command_plan["train_cmd"])

for i, cmd in enumerate(cmds, start=1):
    print(f"[exec {i}/{len(cmds)}] {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Training execution block done.")


In [ ]:
# Step 5: Write baseline training artifact (repo + Drive)
from pathlib import Path

ckpt_files = sorted([str(p) for p in Path(BASELINE_CKPT_DIR).expanduser().glob("*.ckpt")])
payload = {
    "run_id": "smart_baseline_train_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "checkpoint_dir": BASELINE_CKPT_DIR,
    "checkpoint_files": ckpt_files,
    "flow_summary": flow_bundle.summary,
    "command_plan": flow_bundle.command_plan,
    "flow_artifacts": flow_bundle.artifacts,
    "notes": [
        "Training-only notebook. Use smart_rollout_simulation_colab.ipynb and smart_checkpoint_eval_colab.ipynb for simulation/evaluation.",
    ],
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_path = Path("experiments/smart-baseline/results/smart_baseline_train_v0.json")
repo_path.parent.mkdir(parents=True, exist_ok=True)
repo_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_path = persist_run_dir / "outputs" / "smart_baseline_train_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_path:", repo_path)
print("drive_path:", drive_path)
print("num_ckpts_found:", len(ckpt_files))
